**© Copyright AIDENTIFY. All rights reserved.**

본 자료는 **멀티캠퍼스 LLM 파인튜닝 과정** 수강생을 위해 제작되었으며, 강의 목적으로만 사용 가능합니다.  
무단 복제, 배포, 상업적 이용을 금지합니다.

---

# 🎯 Session 6: Transformers 라이브러리 & HuggingFace Hub

## 📋 학습 목표

1️⃣ HuggingFace Hub에 연결하고 모델을 탐색하는 방법을 배운다  
2️⃣ AutoModel/AutoTokenizer로 모델을 로딩하고 추론한다  
3️⃣ Pipeline API를 사용한 간편 추론을 실습한다  
4️⃣ BitsAndBytes 4bit 양자화로 메모리를 절약한다  
5️⃣ 모델 크기별 GPU 메모리 사용량을 비교한다  

---

### 🖥️ 실습 환경
- **GPU**: NVIDIA RTX 4060 (8GB VRAM)
- **기본 모델**: Qwen2.5-1.5B-Instruct
- **필수 패키지**: transformers, torch, accelerate, bitsandbytes

## 1️⃣ GPU 환경 점검 및 유틸리티

In [6]:
import torch, gc

def print_gpu_memory(tag=""):
    """GPU 메모리 사용량을 출력하는 유틸리티 함수"""
    if torch.cuda.is_available():
        allocated = torch.cuda.memory_allocated() / 1024**3
        total = torch.cuda.get_device_properties(0).total_memory / 1024**3
        print(f"[{tag}] GPU: {allocated:.1f}GB / {total:.1f}GB")
    else:
        print("⚠️ GPU를 사용할 수 없습니다.")

def cleanup_gpu():
    """GPU 메모리를 정리합니다."""
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
    print_gpu_memory("정리 후")

# GPU 정보 확인
if torch.cuda.is_available():
    print(f"✅ GPU 이름: {torch.cuda.get_device_name(0)}")
    props = torch.cuda.get_device_properties(0)
    print(f"📊 VRAM: {props.total_memory / 1024**3:.1f}GB")
    print(f"📊 CUDA Capability: {props.major}.{props.minor}")
    print_gpu_memory("초기 상태")
else:
    print("⚠️ CUDA GPU를 찾을 수 없습니다.")

✅ GPU 이름: NVIDIA GeForce RTX 2070
📊 VRAM: 7.8GB
📊 CUDA Capability: 7.5
[초기 상태] GPU: 0.0GB / 7.8GB


## 2️⃣ 필수 패키지 확인

Transformers 생태계의 핵심 패키지들을 확인합니다.

In [7]:
# 📦 패키지 버전 확인
import importlib

packages = [
    "transformers",
    "torch",
    "accelerate",
    "bitsandbytes",
    "huggingface_hub",
]

print("📦 패키지 버전 확인")
print("=" * 40)

for pkg_name in packages:
    try:
        pkg = importlib.import_module(pkg_name)
        version = getattr(pkg, "__version__", "unknown")
        print(f"  ✅ {pkg_name}: {version}")
    except ImportError:
        print(f"  ❌ {pkg_name}: 설치되지 않음")
        print(f"     pip install {pkg_name}")

📦 패키지 버전 확인
  ✅ transformers: 4.57.2
  ✅ torch: 2.10.0+cu128
  ✅ accelerate: 1.13.0
  ✅ bitsandbytes: 0.49.2
  ✅ huggingface_hub: 0.36.2


## 3️⃣ HuggingFace Hub 연결

HuggingFace Hub은 모델, 데이터셋, Space를 공유하는 플랫폼입니다.

### 🔑 Hub 기능
- 🔹 수만 개의 사전학습 모델 제공
- 🔹 모델 카드로 성능/사용법 확인
- 🔹 API 토큰으로 비공개 모델 접근
- 🔹 모델 자동 다운로드 및 캐싱

In [8]:
from huggingface_hub import HfApi, list_models

# 🔍 HuggingFace Hub에서 모델 검색
api = HfApi()

print("🔍 HuggingFace Hub에서 Qwen2.5 모델 검색")
print("=" * 60)

models = list(api.list_models(
    search="Qwen2.5",
    sort="downloads",
    limit=10,
))

for i, model in enumerate(models, 1):
    downloads = model.downloads if hasattr(model, 'downloads') else 'N/A'
    print(f"  {i}. {model.id}")
    print(f"     📥 다운로드: {downloads:,}")
    print()

🔍 HuggingFace Hub에서 Qwen2.5 모델 검색
  1. Qwen/Qwen2.5-7B-Instruct
     📥 다운로드: 12,433,595

  2. Qwen/Qwen2.5-1.5B-Instruct
     📥 다운로드: 10,591,422

  3. Qwen/Qwen2.5-3B-Instruct
     📥 다운로드: 10,072,564

  4. Qwen/Qwen2.5-VL-7B-Instruct
     📥 다운로드: 8,869,992

  5. Qwen/Qwen2.5-0.5B-Instruct
     📥 다운로드: 5,872,425

  6. Qwen/Qwen2.5-VL-3B-Instruct
     📥 다운로드: 5,866,255

  7. Qwen/Qwen2.5-32B-Instruct
     📥 다운로드: 3,038,981

  8. Qwen/Qwen2.5-14B-Instruct-AWQ
     📥 다운로드: 1,881,039

  9. Qwen/Qwen2.5-Coder-7B-Instruct
     📥 다운로드: 1,863,335

  10. Qwen/Qwen2.5-14B-Instruct
     📥 다운로드: 1,858,028



In [9]:
# 🔐 HuggingFace 로그인 (선택사항)
# 비공개 모델이나 게이티드 모델에 접근할 때 필요합니다.

# 방법 1: 환경변수로 설정
# import os
# os.environ["HF_TOKEN"] = "hf_your_token_here"

# 방법 2: huggingface-cli login
# !huggingface-cli login

# 방법 3: 코드에서 직접
# from huggingface_hub import login
# login(token="hf_your_token_here")

print("📌 HuggingFace 토큰 설정 방법 안내:")
print("  1. https://huggingface.co/settings/tokens 에서 토큰 발급")
print("  2. 위 코드에서 원하는 방법으로 설정")
print("  3. Qwen2.5 모델은 공개 모델이므로 토큰 없이도 사용 가능")

📌 HuggingFace 토큰 설정 방법 안내:
  1. https://huggingface.co/settings/tokens 에서 토큰 발급
  2. 위 코드에서 원하는 방법으로 설정
  3. Qwen2.5 모델은 공개 모델이므로 토큰 없이도 사용 가능


## 4️⃣ AutoModel / AutoTokenizer로 모델 로딩

Transformers 라이브러리의 `Auto` 클래스를 사용하면, 모델 이름만으로 자동으로 적절한 클래스를 선택합니다.

### 📚 주요 Auto 클래스
- `AutoModelForCausalLM` - 텍스트 생성 (GPT 계열)
- `AutoModelForSeq2SeqLM` - 시퀀스-투-시퀀스 (T5 계열)
- `AutoTokenizer` - 토크나이저 자동 선택

In [10]:
from transformers import AutoModelForCausalLM, AutoTokenizer
import time

MODEL_NAME = "Qwen/Qwen2.5-1.5B-Instruct"

print(f"📥 모델 로딩 시작: {MODEL_NAME}")
print("⏳ 처음 실행 시 다운로드에 시간이 걸립니다...")
print_gpu_memory("로딩 전")

start_time = time.time()

# 1. 토크나이저 로딩
print("\n🔤 토크나이저 로딩 중...")
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
print(f"  ✅ 토크나이저 로딩 완료")
print(f"  📊 어휘 크기: {tokenizer.vocab_size:,}")

# 2. 모델 로딩 (FP16)
print("\n🧠 모델 로딩 중 (FP16)...")
model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    torch_dtype=torch.float16,
    device_map="auto",
)

elapsed = time.time() - start_time
print(f"  ✅ 모델 로딩 완료! ({elapsed:.1f}초)")
print_gpu_memory("모델 로딩 후")

# 모델 정보 출력
total_params = sum(p.numel() for p in model.parameters())
print(f"\n📊 모델 정보:")
print(f"  파라미터 수: {total_params:,} ({total_params/1e9:.2f}B)")
print(f"  모델 타입: {model.config.model_type}")
print(f"  히든 크기: {model.config.hidden_size}")
print(f"  레이어 수: {model.config.num_hidden_layers}")
print(f"  어텐션 헤드 수: {model.config.num_attention_heads}")

📥 모델 로딩 시작: Qwen/Qwen2.5-1.5B-Instruct
⏳ 처음 실행 시 다운로드에 시간이 걸립니다...
[로딩 전] GPU: 0.0GB / 7.8GB

🔤 토크나이저 로딩 중...
  ✅ 토크나이저 로딩 완료
  📊 어휘 크기: 151,643

🧠 모델 로딩 중 (FP16)...


`torch_dtype` is deprecated! Use `dtype` instead!
Some parameters are on the meta device because they were offloaded to the cpu.


  ✅ 모델 로딩 완료! (3.9초)
[모델 로딩 후] GPU: 1.2GB / 7.8GB

📊 모델 정보:
  파라미터 수: 1,543,714,304 (1.54B)
  모델 타입: qwen2
  히든 크기: 1536
  레이어 수: 28
  어텐션 헤드 수: 12


## 5️⃣ 직접 추론하기 (Manual Generation)

토크나이저와 모델을 직접 사용하여 텍스트를 생성합니다.

In [11]:
# 🔤 토크나이저 동작 이해하기
text = "인공지능은 미래를 바꿀 것입니다."

print("🔤 토크나이저 동작 이해")
print("=" * 50)
print(f"원본 텍스트: {text}")

# 인코딩
tokens = tokenizer.encode(text)
print(f"\n📊 토큰 ID: {tokens}")
print(f"📊 토큰 수: {len(tokens)}")

# 개별 토큰 확인
print(f"\n🔍 개별 토큰 분석:")
for i, token_id in enumerate(tokens):
    token_str = tokenizer.decode([token_id])
    print(f"  {i}: ID={token_id:6d} → '{token_str}'")

# 디코딩
decoded = tokenizer.decode(tokens)
print(f"\n🔄 디코딩 결과: {decoded}")

🔤 토크나이저 동작 이해
원본 텍스트: 인공지능은 미래를 바꿀 것입니다.

📊 토큰 ID: [31328, 78125, 21329, 66019, 33704, 143005, 18411, 81718, 144447, 130882, 13]
📊 토큰 수: 11

🔍 개별 토큰 분석:
  0: ID= 31328 → '인'
  1: ID= 78125 → '공'
  2: ID= 21329 → '지'
  3: ID= 66019 → '능'
  4: ID= 33704 → '은'
  5: ID=143005 → ' 미래'
  6: ID= 18411 → '를'
  7: ID= 81718 → ' 바'
  8: ID=144447 → '꿀'
  9: ID=130882 → ' 것입니다'
  10: ID=    13 → '.'

🔄 디코딩 결과: 인공지능은 미래를 바꿀 것입니다.


In [12]:
# 🧠 Chat 템플릿을 사용한 추론
print("🧠 Chat 템플릿을 사용한 추론")
print("=" * 50)

messages = [
    {"role": "system", "content": "당신은 유능한 AI 어시스턴트입니다. 간결하게 답변해주세요."},
    {"role": "user", "content": "파이썬의 장점 3가지를 알려주세요."}
]

# Chat 템플릿 적용
text_input = tokenizer.apply_chat_template(
    messages,
    tokenize=False,
    add_generation_prompt=True
)
print(f"📝 적용된 템플릿:\n{text_input}")
print(f"\n{'=' * 50}")

# 토큰화 및 추론
inputs = tokenizer(text_input, return_tensors="pt").to(model.device)
print(f"\n📊 입력 토큰 수: {inputs['input_ids'].shape[1]}")

start_time = time.time()

with torch.no_grad():
    outputs = model.generate(
        **inputs,
        max_new_tokens=256,
        temperature=0.7,
        top_p=0.9,
        do_sample=True,
    )

elapsed = time.time() - start_time

# 생성된 토큰만 디코딩
generated_ids = outputs[0][inputs['input_ids'].shape[1]:]
response = tokenizer.decode(generated_ids, skip_special_tokens=True)

print(f"\n💬 모델 응답:\n{response}")
print(f"\n⏱️ 소요 시간: {elapsed:.2f}초")
print(f"📊 생성 토큰 수: {len(generated_ids)}")
print(f"⚡ 처리 속도: {len(generated_ids)/elapsed:.1f} tokens/sec")
print_gpu_memory("추론 후")

🧠 Chat 템플릿을 사용한 추론
📝 적용된 템플릿:
<|im_start|>system
당신은 유능한 AI 어시스턴트입니다. 간결하게 답변해주세요.<|im_end|>
<|im_start|>user
파이썬의 장점 3가지를 알려주세요.<|im_end|>
<|im_start|>assistant



📊 입력 토큰 수: 47

💬 모델 응답:
1. 풍부한 패키지와 라이브러리: 파이썬은 다양한 분야에 걸쳐 많은 패키지와 라이브러리를 제공합니다.

2. 넓은 사용자 community: 파이썬은 세계적으로 큰 사용자 community를 가지고 있어, 질문이나 도움을 받는 데 쉽게 도움이 될 수 있습니다.

3. 개방성과 확장성을 강조하는 언어: 파이썬은 코드를 수정하고 재사용할 수 있는 확장성을 강조하며, 이는 프로그래밍에 대한 접근성이 높아집니다.

⏱️ 소요 시간: 54.87초
📊 생성 토큰 수: 138
⚡ 처리 속도: 2.5 tokens/sec
[추론 후] GPU: 1.2GB / 7.8GB


## 6️⃣ Pipeline API를 사용한 간편 추론

`pipeline`은 모델 로딩부터 추론까지 한 줄로 처리하는 고수준 API입니다.

### 📋 주요 Pipeline 타입
- `text-generation` - 텍스트 생성
- `text-classification` - 텍스트 분류
- `question-answering` - 질의응답
- `summarization` - 요약
- `translation` - 번역

In [13]:
from transformers import pipeline

# 🔧 이미 로딩된 모델로 Pipeline 생성
print("🔧 Pipeline 생성 중...")

pipe = pipeline(
    "text-generation",
    model=model,
    tokenizer=tokenizer,
)

print("✅ Pipeline 생성 완료!")

# Pipeline으로 추론
print("\n💬 Pipeline 추론 테스트")
print("=" * 50)

messages = [
    {"role": "system", "content": "간결하게 한국어로 답변하세요."},
    {"role": "user", "content": "딥러닝이란 무엇인가요?"}
]

start_time = time.time()

result = pipe(
    messages,
    max_new_tokens=200,
    temperature=0.7,
    do_sample=True, # 만약 do_sample=False로 설정하면 가장 확률이 높은 토큰만 선택하여 더 결정론적인 출력을 생성합니다 (Greedy)
)

elapsed = time.time() - start_time

response_text = result[0]["generated_text"][-1]["content"]
print(f"📝 응답:\n{response_text}")
print(f"\n⏱️ 소요 시간: {elapsed:.2f}초")

Device set to use cuda:0


🔧 Pipeline 생성 중...
✅ Pipeline 생성 완료!

💬 Pipeline 추론 테스트
📝 응답:
딥러닝은 컴퓨터 시스템이 데이터를 통해 학습하고, 이를 기반으로 의사 결정을 내리는 방법입니다. 이는 인공지능(AI)의 중요한 분야 중 하나로, 인간의 지능과 비슷한 능력을 가진 AI를 구현하는데 사용됩니다.

⏱️ 소요 시간: 24.58초


In [14]:
# 📝 다양한 질문 테스트
test_questions = [
    "한국의 4계절 특징을 간단히 설명해주세요.",
    "Python에서 리스트와 튜플의 차이점은?",
    "3 + 5 * 2 의 결과는 무엇인가요?",
]

print("📝 다양한 질문 테스트")
print("=" * 60)

for i, question in enumerate(test_questions, 1):
    print(f"\n{'─' * 60}")
    print(f"❓ 질문 {i}: {question}")
    
    messages = [
        {"role": "system", "content": "간결하게 한국어로 답변하세요."},
        {"role": "user", "content": question}
    ]
    
    start_time = time.time()
    result = pipe(messages, max_new_tokens=150, temperature=0.5, do_sample=True)
    elapsed = time.time() - start_time
    
    response_text = result[0]["generated_text"][-1]["content"]
    print(f"💬 응답: {response_text[:200]}")
    print(f"⏱️ {elapsed:.2f}초")

📝 다양한 질문 테스트

────────────────────────────────────────────────────────────
❓ 질문 1: 한국의 4계절 특징을 간단히 설명해주세요.
💬 응답: 한국의 4계절은 다음과 같습니다:

1. 겨울 (12월 ~ 2월): 눈이 많이 내리는 계절입니다.
2. 봄 (3월 ~ 5월): 따뜻하고 꽃들이 피는 계절입니다.
3. 여름 (6월 ~ 8월): 더운 계절이며, 해수욕과 바다 여행이 활발합니다.
4. 가을 (9월 ~ 11월): 날씨가 변하며, 첫눈이 오기도 합니다.

각 계절마다 특별한 변화와 기후 조건이 있습
⏱️ 52.59초

────────────────────────────────────────────────────────────
❓ 질문 2: Python에서 리스트와 튜플의 차이점은?
💬 응답: Python에서 리스트(LIST)와 튜플(TUPLE)은 주로 동일한 기능을 하지만 구조나 사용법에 차이가 있습니다.

1. **구조**:
   - **리스트**: 순서를 가진 자료형입니다.
   - **튜플**: 순서를 가진 자료형이며, 변경 불가능합니다.

2. **변경 가능성**:
   - **리스트**: 변경 가능합니다.
   - **튜플**: 변경 
⏱️ 50.90초

────────────────────────────────────────────────────────────
❓ 질문 3: 3 + 5 * 2 의 결과는 무엇인가요?
💬 응답: 3 + 5 * 2의 결과는 13입니다.
⏱️ 5.98초


In [15]:
# 🧹 FP16 모델 메모리 해제
print("🧹 FP16 모델 메모리 해제")
print_gpu_memory("해제 전")

del model, pipe
cleanup_gpu()

print("✅ FP16 모델이 메모리에서 해제되었습니다.")

🧹 FP16 모델 메모리 해제
[해제 전] GPU: 1.2GB / 7.8GB
[정리 후] GPU: 0.0GB / 7.8GB
✅ FP16 모델이 메모리에서 해제되었습니다.


## 7️⃣ BitsAndBytes 4bit 양자화

**양자화(Quantization)**는 모델의 가중치를 낮은 정밀도로 변환하여 메모리를 절약하는 기술입니다.

### 📊 양자화 방식 비교

| 방식 | 정밀도 | 메모리 | 품질 저하 |
|------|--------|--------|----------|
| FP32 | 32bit | 기준 | 없음 |
| FP16 | 16bit | 50% | 거의 없음 |
| INT8 | 8bit | 25% | 약간 |
| **NF4** | **4bit** | **12.5%** | **약간~보통** |

### 🔑 BitsAndBytes NF4 양자화
- NormalFloat4 (NF4): 정규분포 기반 4bit 양자화
- QLoRA 논문에서 제안된 기법
- double quantization으로 추가 메모리 절약

In [16]:
from transformers import BitsAndBytesConfig

# ⚙️ 4bit 양자화 설정
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,                    # 4bit 양자화 활성화
    bnb_4bit_quant_type="nf4",            # NormalFloat4 타입
    bnb_4bit_compute_dtype=torch.float16,  # 연산 시 FP16 사용
    bnb_4bit_use_double_quant=True,       # 이중 양자화 (추가 메모리 절약)
)

print("⚙️ BitsAndBytes 4bit 양자화 설정:")
print(f"  🔹 양자화 타입: NF4 (NormalFloat4)")
print(f"  🔹 연산 정밀도: FP16")
print(f"  🔹 이중 양자화: 활성화")
print(f"\n📊 예상 메모리 절약:")
print(f"  1.5B 모델: ~3GB (FP16) → ~1GB (4bit)")
print(f"  7B 모델: ~14GB (FP16) → ~4GB (4bit)")

⚙️ BitsAndBytes 4bit 양자화 설정:
  🔹 양자화 타입: NF4 (NormalFloat4)
  🔹 연산 정밀도: FP16
  🔹 이중 양자화: 활성화

📊 예상 메모리 절약:
  1.5B 모델: ~3GB (FP16) → ~1GB (4bit)
  7B 모델: ~14GB (FP16) → ~4GB (4bit)


In [17]:
# 📥 4bit 양자화 모델 로딩
print(f"📥 4bit 양자화 모델 로딩: {MODEL_NAME}")
print_gpu_memory("로딩 전")

start_time = time.time()

tokenizer_4bit = AutoTokenizer.from_pretrained(MODEL_NAME)

model_4bit = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    quantization_config=bnb_config,
    device_map="auto",
)

elapsed = time.time() - start_time
print(f"\n✅ 4bit 모델 로딩 완료! ({elapsed:.1f}초)")
print_gpu_memory("4bit 로딩 후")

# 모델 정보
total_params = sum(p.numel() for p in model_4bit.parameters())
print(f"\n📊 모델 정보:")
print(f"  파라미터 수: {total_params:,}")
print(f"  양자화 상태: 4bit (NF4)")

📥 4bit 양자화 모델 로딩: Qwen/Qwen2.5-1.5B-Instruct
[로딩 전] GPU: 0.0GB / 7.8GB

✅ 4bit 모델 로딩 완료! (4.9초)
[4bit 로딩 후] GPU: 1.1GB / 7.8GB

📊 모델 정보:
  파라미터 수: 888,616,448
  양자화 상태: 4bit (NF4)


In [18]:
# 🔍 4bit 모델 추론 테스트
print("🔍 4bit 양자화 모델 추론 테스트")
print("=" * 50)

messages = [
    {"role": "system", "content": "간결하게 한국어로 답변하세요."},
    {"role": "user", "content": "파이썬의 장점 3가지를 알려주세요."}
]

text_input = tokenizer_4bit.apply_chat_template(
    messages, tokenize=False, add_generation_prompt=True
)
inputs = tokenizer_4bit(text_input, return_tensors="pt").to(model_4bit.device)

start_time = time.time()

with torch.no_grad():
    outputs = model_4bit.generate(
        **inputs,
        max_new_tokens=200,
        temperature=0.7,
        do_sample=True,
    )

elapsed = time.time() - start_time

generated_ids = outputs[0][inputs['input_ids'].shape[1]:]
response = tokenizer_4bit.decode(generated_ids, skip_special_tokens=True)

print(f"💬 4bit 모델 응답:\n{response}")
print(f"\n⏱️ 소요 시간: {elapsed:.2f}초")
print(f"📊 생성 토큰 수: {len(generated_ids)}")
print(f"⚡ 처리 속도: {len(generated_ids)/elapsed:.1f} tokens/sec")
print_gpu_memory("4bit 추론 후")

🔍 4bit 양자화 모델 추론 테스트
💬 4bit 모델 응답:
1. 편집성: 파이썬은 간단한 코드로도 복잡한 프로그램을 작성할 수 있어, 개발자가 코드를 쉽게 수정하고 재사용할 수 있게 해줍니다.
2. 확장성: 파이썬은 다양한 모듈과 패키지들을 제공하여, 새로운 기능이나 도구를 쉽게 추가하거나 확장할 수 있습니다.
3. 접근성: 파이썬은 무료이고 공식적인 라이선스나 가격 없이 사용할 수 있으며, 이는 모든 사람들이 파이썬의 가능성을 배울 수 있다는 것을 의미합니다.

⏱️ 소요 시간: 9.75초
📊 생성 토큰 수: 136
⚡ 처리 속도: 14.0 tokens/sec
[4bit 추론 후] GPU: 1.1GB / 7.8GB


## 8️⃣ FP16 vs 4bit 메모리 비교

같은 모델의 FP16과 4bit 버전의 GPU 메모리 사용량을 비교합니다.

In [19]:
# 📊 FP16 vs 4bit 비교 결과 정리
# (위 실습에서 측정된 값을 기반으로 비교합니다)

print("📊 Qwen2.5-1.5B-Instruct 메모리 비교")
print("=" * 55)
print(f"{'항목':<15} {'FP16':>12} {'4bit (NF4)':>12} {'절약률':>10}")
print("-" * 55)

# 이론적 값 (실습 결과에 따라 다를 수 있음)
comparisons = [
    ("모델 크기", "~3.0 GB", "~1.0 GB", "~67%"),
    ("추론 시 피크", "~3.5 GB", "~1.5 GB", "~57%"),
    ("로딩 시간", "~5초", "~10초", "느림"),
    ("추론 속도", "빠름", "약간 느림", "-"),
    ("품질", "기준", "약간 저하", "-"),
]

for item, fp16, q4, saving in comparisons:
    print(f"{item:<15} {fp16:>12} {q4:>12} {saving:>10}")

print(f"\n💡 핵심 포인트:")
print(f"  🔹 4bit 양자화는 메모리를 60~70% 절약")
print(f"  🔹 품질 저하는 대부분의 작업에서 미미")
print(f"  🔹 RTX 4060 (8GB)에서 7B 모델 사용 가능")
print(f"  🔹 로딩 시간은 양자화로 인해 약간 증가")

📊 Qwen2.5-1.5B-Instruct 메모리 비교
항목                      FP16   4bit (NF4)        절약률
-------------------------------------------------------
모델 크기                ~3.0 GB      ~1.0 GB       ~67%
추론 시 피크              ~3.5 GB      ~1.5 GB       ~57%
로딩 시간                    ~5초         ~10초         느림
추론 속도                     빠름        약간 느림          -
품질                        기준        약간 저하          -

💡 핵심 포인트:
  🔹 4bit 양자화는 메모리를 60~70% 절약
  🔹 품질 저하는 대부분의 작업에서 미미
  🔹 RTX 4060 (8GB)에서 7B 모델 사용 가능
  🔹 로딩 시간은 양자화로 인해 약간 증가


In [20]:
# 🧹 4bit 모델 메모리 해제
print("🧹 4bit 모델 메모리 해제")
del model_4bit, tokenizer_4bit
cleanup_gpu()

🧹 4bit 모델 메모리 해제
[정리 후] GPU: 0.0GB / 7.8GB


## 9️⃣ 생성 파라미터 이해

`model.generate()`에서 사용하는 주요 파라미터를 정리합니다.

In [21]:
# 📖 생성 파라미터 정리
params = [
    ("max_new_tokens", "생성할 최대 토큰 수", "128~512"),
    ("temperature", "랜덤성 조절 (높을수록 다양)", "0.1~1.0"),
    ("top_p", "누적 확률 기반 샘플링", "0.9~0.95"),
    ("top_k", "상위 k개 토큰에서 샘플링", "20~50"),
    ("repetition_penalty", "반복 억제 (1.0=없음)", "1.0~1.3"),
    ("do_sample", "샘플링 활성화 (False=greedy)", "True/False"),
    ("num_beams", "빔 서치 빔 수", "1~5"),
]

print("📖 텍스트 생성 주요 파라미터")
print("=" * 65)
print(f"{'파라미터':<25} {'설명':<25} {'권장값':>12}")
print("-" * 65)
for name, desc, val in params:
    print(f"{name:<25} {desc:<25} {val:>12}")

print(f"\n💡 팁:")
print(f"  🔹 temperature=0: 항상 같은 결과 (결정적)")
print(f"  🔹 temperature=1: 다양한 결과 (창의적)")
print(f"  🔹 코드 생성: temperature=0.2, 대화: temperature=0.7")

📖 텍스트 생성 주요 파라미터
파라미터                      설명                                 권장값
-----------------------------------------------------------------
max_new_tokens            생성할 최대 토큰 수                    128~512
temperature               랜덤성 조절 (높을수록 다양)               0.1~1.0
top_p                     누적 확률 기반 샘플링                  0.9~0.95
top_k                     상위 k개 토큰에서 샘플링                   20~50
repetition_penalty        반복 억제 (1.0=없음)                 1.0~1.3
do_sample                 샘플링 활성화 (False=greedy)      True/False
num_beams                 빔 서치 빔 수                           1~5

💡 팁:
  🔹 temperature=0: 항상 같은 결과 (결정적)
  🔹 temperature=1: 다양한 결과 (창의적)
  🔹 코드 생성: temperature=0.2, 대화: temperature=0.7


In [22]:
# 🧹 최종 GPU 메모리 확인
cleanup_gpu()

print("\n📌 오늘의 핵심 정리:")
print("  1️⃣ AutoModelForCausalLM으로 LLM을 쉽게 로딩할 수 있다")
print("  2️⃣ apply_chat_template으로 올바른 프롬프트 포맷을 만든다")
print("  3️⃣ Pipeline API는 추론을 한 줄로 단순화한다")
print("  4️⃣ 4bit 양자화로 GPU 메모리를 60~70% 절약한다")
print("  5️⃣ 생성 파라미터로 출력 품질과 다양성을 조절한다")

[정리 후] GPU: 0.0GB / 7.8GB

📌 오늘의 핵심 정리:
  1️⃣ AutoModelForCausalLM으로 LLM을 쉽게 로딩할 수 있다
  2️⃣ apply_chat_template으로 올바른 프롬프트 포맷을 만든다
  3️⃣ Pipeline API는 추론을 한 줄로 단순화한다
  4️⃣ 4bit 양자화로 GPU 메모리를 60~70% 절약한다
  5️⃣ 생성 파라미터로 출력 품질과 다양성을 조절한다


---

## 🎯 실습 과제

1️⃣ `temperature`를 0.1, 0.5, 1.0으로 바꿔가며 같은 질문에 대한 응답 차이를 비교해보세요  
2️⃣ FP16과 4bit 모델에 같은 질문을 해서 품질 차이를 체감해보세요  
3️⃣ `Qwen/Qwen2.5-3B-Instruct` 모델을 4bit로 로딩해보고 메모리를 확인하세요  

---

## 📚 참고 자료
- [HuggingFace Transformers 문서](https://huggingface.co/docs/transformers)
- [BitsAndBytes 문서](https://github.com/TimDettmers/bitsandbytes)
- [Qwen2.5 모델 카드](https://huggingface.co/Qwen/Qwen2.5-1.5B-Instruct)